In [1]:
import torch 
import sys
from datasets.graph_datasets.graph_heat_dataset import HeatGraphDataset
import yaml
from models.forecasting.GNO import GNO
from torch_geometric.loader import DataLoader
from datasets.graph_datasets.graph_data_utils import partition_domain_into_subgraphs
sys.path.append('...')

/Users/louisgodtfredsen/Desktop/Coding Projects/ML-for-PDEs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load configs
with open("../checkpoints/gno_heat/20260912_2148/model_configs.yaml", "r") as file:
    cfg = yaml.safe_load(file)

# Load in sample to run inference on

In [14]:
data_path ='../data/test_data/heat_equation_m64_h0_minmax_N200.pt'
input_data = torch.load(data_path)
simulation_idx = 0
simulation_frame = 0
field_keys = list(input_data.keys())[1:]

X, X_t1 = input_data['X'][simulation_idx,simulation_frame], input_data['X'][simulation_idx,simulation_frame+1]
pde_params = [float(input_data[k][simulation_idx]) for k in field_keys]

H, W = X.shape[-1], X.shape[-2] 
x_indices = torch.tensor([x for x in range(W)])
y_indices = torch.tensor([x for x in range(H)])
node_grid_indices = torch.cartesian_prod(x_indices, y_indices) # Collection of (x, y) grid indices
node_spatial_pos = torch.cartesian_prod(x_indices / W, y_indices / H) # Collection of (x, y) spatial positions

dataset = HeatGraphDataset('../data/test_data/heat_equation_m64_h0_minmax_N200.pt',
                           list(input_data.keys())[1:],
                           r = cfg['radius'],
                           bc ='periodic',
                           sub_graph_size = cfg['sub_graph_size'])

In [19]:
subgraphs = partition_domain_into_subgraphs(X, X_t1, H, W, num_subgraph_nodes=200, r = 0.025, boundary_condition='periodic', pde_params=pde_params)

In [20]:
subgraphs[0]

Data(x=[200, 3], edge_index=[2, 1424], edge_attr=[1424, 5], y=[200, 1], num_nodes=200)

# Load Model

In [ ]:
gno_model = GNO(optimiser = cfg['optimiser'], 
                 learning_rate = cfg['learning_rate'], 
                 num_node_input_features = cfg['num_node_input_features'],
                 num_edge_features = cfg['num_edge_features'], 
                 num_latent_dim = cfg['num_latent_dim'], 
                 output_dim = cfg['output_dim'],
                 num_gno_layers = cfg['num_gno_layers'],
                 kernel_ffn_layers = cfg['kernel_ffn_layers'],
                 kernel_ffn_dropout = cfg['kernel_ffn_dropout'], 
                 GNO_layer_activation = cfg['gno_layer_activation'])

model_path = '../checkpoints/gno_heat/20260912_2148/gno-epoch=0009-val_loss=0.0000.ckpt'
gno_model.load_state_dict(torch.load(model_path)['state_dict'])

<All keys matched successfully>

In [ ]:
import torch

A = torch.randint(low = 0, high=5, size = (5,5))

In [ ]:
A

tensor([[0, 1, 1, 0, 4],
        [4, 2, 3, 4, 0],
        [2, 4, 3, 4, 1],
        [3, 2, 2, 4, 1],
        [4, 0, 4, 2, 1]])

In [ ]:
A[0:3, 1:4]

tensor([[1, 1, 0],
        [2, 3, 4],
        [4, 3, 4]])

In [ ]:
A.flatten()

tensor([0, 1, 1, 0, 4, 4, 2, 3, 4, 0, 2, 4, 3, 4, 1, 3, 2, 2, 4, 1, 4, 0, 4, 2,
        1])